# UTTOPIA Prediction Script

Loads a pre-trained Random Forest model and PCA transformer to predict
protein complex properties (Kd or stoichiometry) from user-provided data.

In [ ]:
import pandas as pd
import numpy as np
import ast
import joblib
import os
from huggingface_hub import hf_hub_download

## User Configuration

In [ ]:
ORGANISM        = "Human" # Choose: "Human" or "Yeast"
TARGET          = "Kd"    # Choose: "Kd" or "int_stoich"

BASE_PATH       = os.getcwd()
USER_INPUT_FILE = os.path.join(BASE_PATH, f"input_to_RF_{ORGANISM}.csv")
OUTPUT_FILE     = os.path.join(BASE_PATH, f"UTTOPIA_Predictions_{ORGANISM}_{TARGET}.csv")

HF_REPO         = "caioo61/UTTOPIA-RF-ESM"

## Configuration & Path

In [ ]:
def download_from_hf(filename):
    """ Download from Hugging Face """
    return hf_hub_download(
        repo_id = HF_REPO,
        filename = filename,
        repo_type = "dataset",
        local_dir = BASE_PATH,    
        local_dir_use_symlinks = False
    )

In [ ]:
CONFIG = {
    f"{ORGANISM}": {
        "esm_db": f"Dataset/{ORGANISM}/Dataset_pca_esm_{ORGANISM}.csv",
        "pca_model": f"joblib/{ORGANISM}/uttopia_pca_esm_feat_{ORGANISM}.joblib",
        "pca_prefix": "ESM_pca", 
        "rf_models": {
            "Kd": f"joblib/{ORGANISM}/uttopia_rf_{ORGANISM}_KD.joblib",
            "int_stoich": f"joblib/{ORGANISM}/uttopia_rf_{ORGANISM}_INT_STOICH.joblib"
        }
    }
}
        
# Validate user inputs
if ORGANISM not in CONFIG:
    raise ValueError(f"Invalid Organism '{ORGANISM}'. Please choose 'Human' or 'Yeast'.")
if TARGET not in ["Kd", "int_stoich"]:
    raise ValueError(f"Invalid Target '{TARGET}'. Please choose 'Kd' or 'int_stoich'.")

## Loading Model & Databases

In [ ]:
org_config = CONFIG[ORGANISM]

print("Downloading files from Hugging Face")
rf_model = joblib.load(download_from_hf(org_config["rf_models"][TARGET]))
pca_model = joblib.load(download_from_hf(org_config["pca_model"]))
esm_db = pd.read_csv(download_from_hf(org_config["esm_db"]))
print("Files downloaded :)")

def parse_vector(x):
    """ Safely evaluates ESM vector strings into numpy arrays """
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=np.float32)
    return x

esm_db['esm_vector'] = esm_db['esm_vector'].apply(parse_vector)
esm_dict = dict(zip(esm_db['gene_name'].astype(str), esm_db['esm_vector']))


## User Data Processing

In [ ]:
user_df = pd.read_csv(USER_INPUT_FILE)

# Quick check for missing proteins
missing = set(user_df['Cmin'].astype(str)).union(user_df['Cmax'].astype(str)) - set(esm_dict.keys())

if missing:
    print(f"\nWarning: Found {len(missing)} unknown proteins. Discarding invalid rows...")
    print(f"List of proteins not found: {', '.join(missing)}") 
    user_df = user_df[~user_df['Cmin'].astype(str).isin(missing) & ~user_df['Cmax'].astype(str).isin(missing)].copy()
    if user_df.empty: 
        raise ValueError("Fatal Error: No valid data left to predict.")
    
# Extract ESM vectors (with a fallback of zeros for safety)
fallback = np.zeros(320, dtype=np.float32) 
cmin_vectors = np.vstack([esm_dict.get(str(g), fallback) for g in user_df["Cmin"]])
cmax_vectors = np.vstack([esm_dict.get(str(g), fallback) for g in user_df["Cmax"]])

# Apply PCA compression (fixed to 20 components as trained)
cmin_pca = pca_model.transform(cmin_vectors)
cmax_pca = pca_model.transform(cmax_vectors)

# Generate PCA column names dynamically based on the organism (ESM_pca vs pca)
pref = org_config["pca_prefix"]
pca_cols = ([f"Cmin_{pref}_{i+1}" for i in range(20)] + [f"Cmax_{pref}_{i+1}" for i in range(20)])
pca_df = pd.DataFrame(np.hstack([cmin_pca, cmax_pca]), columns=pca_cols, index=user_df.index)

# Assemble final feature matrix
base_cols = [col for col in user_df.columns if col not in ['Cmin', 'Cmax']]
X_test = pd.concat([user_df[base_cols], pca_df], axis=1)

# Ensure column order strictly matches the training phase
try:
    X_test = X_test[rf_model.feature_names_in_]
except KeyError as e:
    raise KeyError(f"Missing biological features required by the model. Details: {e}")


## Prediction

In [ ]:
y_pred = rf_model.predict(X_test)
user_df[f'Predicted_{TARGET}'] = y_pred

user_df.to_csv(OUTPUT_FILE, index=False)
print(f"Done! Results saved to: '{OUTPUT_FILE}' :)")